In [105]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import warnings 
warnings.filterwarnings("ignore")

In [106]:
df = pd.read_csv("data/gemstone.csv")
df.head()

,id,carat,cut,color,clarity,depth,table,x,y,z,price
0,0,1.52,Premium,F,VS2,62.2,58.0,7.27,7.33,4.55,13619
1,1,2.03,Very Good,J,SI2,62.0,58.0,8.06,8.12,5.05,13387
2,2,0.70,Ideal,G,VS1,61.2,57.0,5.69,5.73,3.50,2772
3,3,0.32,Ideal,G,VS1,61.6,56.0,4.38,4.41,2.71,666
4,4,1.70,Premium,G,VS2,62.6,59.0,7.65,7.61,4.77,14453


In [107]:
df = df.drop("id", axis=1)
df.head()

,carat,cut,color,clarity,depth,table,x,y,z,price
0,1.52,Premium,F,VS2,62.2,58.0,7.27,7.33,4.55,13619
1,2.03,Very Good,J,SI2,62.0,58.0,8.06,8.12,5.05,13387
2,0.70,Ideal,G,VS1,61.2,57.0,5.69,5.73,3.50,2772
3,0.32,Ideal,G,VS1,61.6,56.0,4.38,4.41,2.71,666
4,1.70,Premium,G,VS2,62.6,59.0,7.65,7.61,4.77,14453


In [108]:
X = df.drop("price", axis=1)
y = df["price"]

In [109]:
categorical_features = X.select_dtypes(include=["object"]).columns
numerical_features = X.select_dtypes(exclude=["object"]).columns

# Define the custom ranking for each ordinal variable
cut_categories = ['Fair', 'Good', 'Very Good','Premium','Ideal']
color_categories = ['D', 'E', 'F', 'G', 'H', 'I', 'J']
clarity_categories = ['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF']


num_pipline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ('ordinal_encoder',OrdinalEncoder(categories=[cut_categories,color_categories,clarity_categories])),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipline, numerical_features),
    ("cat", cat_pipline, categorical_features)
])

In [110]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [111]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

In [112]:
def model_evaluation(true, pred):
    mse = mean_squared_error(true, pred)
    r2 = r2_score(true, pred)
    return mse, r2

In [113]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "K-Nearest Neighbors": KNeighborsRegressor(),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "xgboost": XGBRegressor(random_state=42),
    "catboost": CatBoostRegressor(random_state=42, verbose=0)
}

model_list = []
r2_score_list = []

for name, model in models.items():
    

    model.fit(X_train_preprocessed, y_train)

    y_train_pred = model.predict(X_train_preprocessed)
    y_test_pred = model.predict(X_test_preprocessed)

    train_mse, train_r2 = model_evaluation(y_train, y_train_pred)
    test_mse, test_r2 = model_evaluation(y_test, y_test_pred)

    print(f'{name}Model performance for Training set')
    print("- MSE: {:.4f}".format(train_mse))
    print("- R2 Score: {:.4f}".format(train_r2))

    print(f'{name}Model performance for Test set')
    print("- MSE: {:.4f}".format(test_mse))
    print("- R2 Score: {:.4f}".format(test_r2))

    model_list.append(name)
    r2_score_list.append(test_r2)


    print('='*35)
    print('\n')

Linear RegressionModel performance for Training set
- MSE: 1034185.3253
- R2 Score: 0.9366
Linear RegressionModel performance for Test set
- MSE: 1013245.5453
- R2 Score: 0.9373


Random ForestModel performance for Training set
- MSE: 52062.0476
- R2 Score: 0.9968
Random ForestModel performance for Test set
- MSE: 368680.8919
- R2 Score: 0.9772


Gradient BoostingModel performance for Training set
- MSE: 375301.5061
- R2 Score: 0.9770
Gradient BoostingModel performance for Test set
- MSE: 385659.9091
- R2 Score: 0.9761


Ridge RegressionModel performance for Training set
- MSE: 1034185.4419
- R2 Score: 0.9366
Ridge RegressionModel performance for Test set
- MSE: 1013256.1052
- R2 Score: 0.9373


Lasso RegressionModel performance for Training set
- MSE: 1034434.9530
- R2 Score: 0.9366
Lasso RegressionModel performance for Test set
- MSE: 1013790.3799
- R2 Score: 0.9373


K-Nearest NeighborsModel performance for Training set
- MSE: 297738.0849
- R2 Score: 0.9817
K-Nearest NeighborsModel 

In [114]:
df_results = pd.DataFrame(list(zip(model_list, r2_score_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)
df_results

,Model Name,R2_Score
8,catboost,0.979317
7,xgboost,0.978790
1,Random Forest,0.977185
2,Gradient Boosting,0.976134
5,K-Nearest Neighbors,0.972114
0,Linear Regression,0.937298
3,Ridge Regression,0.937297
4,Lasso Regression,0.937264
6,AdaBoost,0.862312


### tuning Catboost model

In [115]:
# Initializing catboost
cbr = CatBoostRegressor(verbose=False)

# Creating the hyperparameter grid
param_dist = {'depth'          : [4,5,6,7,8,9, 10],
              'learning_rate' : [0.01,0.02,0.03,0.04],
               'iterations'    : [300,400,500,600]}

#Instantiate RandomSearchCV object
rscv = RandomizedSearchCV(cbr , param_dist, scoring='r2', cv =5, n_jobs=-1)

# Fit the model
rscv.fit(X_train_preprocessed, y_train)

# Print the tuned parameters and score
print(rscv.best_params_)
print(rscv.best_score_)

{'learning_rate': 0.04, 'iterations': 400, 'depth': 10}
0.9797613771610154


In [116]:
# Initializing xgboost
xgb = XGBRegressor()

# Parameters
params = {
 'learning_rate' : [0.05,0.10,0.15,0.20,0.25,0.30],
 'max_depth' : [ 3, 4, 5, 6, 8, 10, 12, 15],
 'min_child_weight' : [ 1, 3, 5, 7 ],
 'gamma': [ 0.0, 0.1, 0.2 , 0.3, 0.4 ],
 'colsample_bytree' : [ 0.3, 0.4, 0.5 , 0.7 ],
 'n_estimators':[300,400,500,600]
}

rs_xgb=RandomizedSearchCV(xgb,param_distributions=params,scoring='r2',n_jobs=-1,cv=5)
rs_xgb.fit(X_train_preprocessed, y_train)

# Print the tuned parameters and score
print(rs_xgb.best_params_)
print(rs_xgb.best_score_)

{'n_estimators': 400, 'min_child_weight': 7, 'max_depth': 5, 'learning_rate': 0.25, 'gamma': 0.0, 'colsample_bytree': 0.4}
0.9791164398193359
